In [1]:
from pathlib import Path
import pandas as pd

# Raiz do projeto = pasta pai de 'notebooks/'
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "finbra.parquet"

df = pd.read_parquet(DATA_PROCESSED)
print(f"Carregado: {len(df):,} linhas, {df['ano'].nunique()} anos")

Carregado: 50,334 linhas, 6 anos


In [2]:
df.groupby("ano")["Instituição"].nunique().rename("capitais_reportadas")

ano
2020    26
2021    26
2022    26
2023    26
2024    26
2025    11
Name: capitais_reportadas, dtype: int64

In [3]:
df[df["tipo_conta"] == "subfuncao"][["funcao", "subfuncao"]].drop_duplicates().head(10)

,funcao,subfuncao
2,01 - Legislativa,01.031 - Ação Legislativa
4,03 - Essencial à Justiça,03.092 - Representação Judicial e Extrajudicial
6,04 - Administração,04.121 - Planejamento e Orçamento
7,04 - Administração,04.122 - Administração Geral
8,04 - Administração,04.123 - Administração Financeira
9,04 - Administração,04.124 - Controle Interno
10,04 - Administração,04.125 - Normatização e Fiscalização
11,04 - Administração,04.126 - Tecnologia da Informação
12,04 - Administração,04.131 - Comunicação Social
14,06 - Segurança Pública,06.181 - Policiamento


In [4]:
import duckdb

# Conecta direto no parquet via DuckDB (sem precisar "injetar" em tabela)
con = duckdb.connect(":memory:")
con.execute(f"CREATE VIEW despesas AS SELECT * FROM read_parquet('{DATA_PROCESSED.as_posix()}')")

query_execucao = """
    WITH base AS (
        SELECT
            "Instituição" AS capital,
            "UF" AS uf,
            funcao,
            "Coluna" AS estagio,
            SUM("Valor") AS valor
        FROM despesas
        WHERE tipo_conta = 'funcao'
          AND ano = 2024
          AND "Coluna" IN ('Despesas Empenhadas', 'Despesas Pagas')
        GROUP BY 1, 2, 3, 4
    )
    SELECT
        capital,
        uf,
        funcao,
        MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END) AS empenhado,
        MAX(CASE WHEN estagio = 'Despesas Pagas' THEN valor END) AS pago,
        ROUND(
            100.0 * MAX(CASE WHEN estagio = 'Despesas Pagas' THEN valor END)
            / NULLIF(MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END), 0),
            1
        ) AS taxa_execucao_pct
    FROM base
    GROUP BY capital, uf, funcao
    ORDER BY taxa_execucao_pct ASC
"""

df_execucao = con.execute(query_execucao).df()
df_execucao.head(15)

,capital,uf,funcao,empenhado,pago,taxa_execucao_pct
0,Prefeitura Municipal de São Luís - MA,MA,23 - Comércio e Serviços,1.867306e+06,5.591641e+05,29.9
1,Prefeitura Municipal de Maceió - AL,AL,16 - Habitação,2.168153e+06,6.496748e+05,30.0
2,Prefeitura Municipal de Teresina - PI,PI,17 - Saneamento,4.825483e+06,1.586615e+06,32.9
3,Prefeitura Municipal de Teresina - PI,PI,18 - Gestão Ambiental,3.913655e+06,1.576163e+06,40.3
4,Prefeitura Municipal de Curitiba - PR,PR,16 - Habitação,4.745052e+07,2.180763e+07,46.0
5,Prefeitura Municipal de Vitória - ES,ES,23 - Comércio e Serviços,1.573912e+07,7.804479e+06,49.6
6,Prefeitura Municipal de São Luís - MA,MA,20 - Agricultura,4.555375e+06,2.291631e+06,50.3
7,Prefeitura Municipal de Porto Alegre - RS,RS,20 - Agricultura,9.954139e+05,5.063479e+05,50.9
8,Prefeitura Municipal de Vitória - ES,ES,24 - Comunicações,9.937871e+06,5.091426e+06,51.2
9,Prefeitura Municipal de Curitiba - PR,PR,15 - Urbanismo,1.805831e+09,9.820598e+08,54.4


In [5]:
import plotly.express as px

# Taxa de execução média das capitais, por função (todas, não só as piores)
media_por_funcao = (
    df_execucao.groupby("funcao")["taxa_execucao_pct"]
    .mean()
    .reset_index()
    .rename(columns={"taxa_execucao_pct": "media_capitais"})
)

# Recorte só de Maceió
maceio = df_execucao[df_execucao["capital"].str.contains("Maceió")][
    ["funcao", "taxa_execucao_pct"]
].rename(columns={"taxa_execucao_pct": "maceio"})

comparativo = media_por_funcao.merge(maceio, on="funcao", how="left").sort_values("funcao")
comparativo["diferenca"] = comparativo["maceio"] - comparativo["media_capitais"]

fig = px.bar(
    comparativo.sort_values("diferenca"),
    x="diferenca",
    y="funcao",
    orientation="h",
    color="diferenca",
    color_continuous_scale=["red", "lightgray", "green"],
    color_continuous_midpoint=0,
    title="Maceió vs. média das capitais — diferença na taxa de execução por função (2024)",
    labels={"diferenca": "Diferença (p.p.)", "funcao": ""},
)
fig.update_layout(height=700)
fig.show()

comparativo.sort_values("diferenca").head(10)

,funcao,media_capitais,maceio,diferenca
15,16 - Habitação,85.166667,30.0,-55.166667
5,06 - Segurança Pública,92.132000,65.9,-26.232000
13,14 - Direitos da Cidadania,88.808333,68.6,-20.208333
14,15 - Urbanismo,88.600000,80.9,-7.700000
11,12 - Educação,92.865385,85.5,-7.365385
12,13 - Cultura,88.469231,82.4,-6.069231
18,19 - Ciência e Tecnologia,93.712500,87.7,-6.012500
7,08 - Assistência Social,92.726923,89.7,-3.026923
25,27 - Desporto e Lazer,89.565385,89.9,0.334615
8,09 - Previdência Social,98.615385,100.0,1.384615


In [6]:
query_habitacao_maceio = """
    SELECT
        subfuncao,
        "Coluna" AS estagio,
        SUM("Valor") AS valor
    FROM despesas
    WHERE "Instituição" LIKE '%Maceió%'
      AND funcao = '16 - Habitação'
      AND ano = 2024
      AND "Coluna" IN ('Despesas Empenhadas', 'Despesas Pagas')
    GROUP BY subfuncao, "Coluna"
    ORDER BY subfuncao
"""

df_habitacao = con.execute(query_habitacao_maceio).df()
df_habitacao

,subfuncao,estagio,valor
0,16.482 - Habitação Urbana,Despesas Empenhadas,2168153.21
1,16.482 - Habitação Urbana,Despesas Pagas,649674.84
2,NaN,Despesas Empenhadas,2168153.21
3,NaN,Despesas Pagas,649674.84


Em 2024, Maceió empenhou R$ 2,17 milhões em Habitação (subfunção 16.482 - Habitação Urbana), mas pagou apenas R$ 649,7 mil — uma taxa de execução de 30%, bem abaixo da média das capitais (85,2%). Diferente de outras funções, o gasto com Habitação em Maceió está concentrado num único programa, o que sugere que a baixa execução está ligada a um projeto/contrato específico que não avançou como planejado no ano, e não a um problema disperso de gestão orçamentária na área.

In [7]:
query_per_capita = """
    SELECT
        "Instituição" AS capital,
        "UF" AS uf,
        funcao,
        MAX("População") AS populacao,
        SUM("Valor") AS valor_pago
    FROM despesas
    WHERE tipo_conta = 'funcao'
      AND ano = 2024
      AND "Coluna" = 'Despesas Pagas'
      AND funcao IN ('10 - Saúde', '12 - Educação')
    GROUP BY capital, uf, funcao
"""

df_per_capita = con.execute(query_per_capita).df()
df_per_capita["per_capita"] = df_per_capita["valor_pago"] / df_per_capita["populacao"]
df_per_capita["destaque"] = df_per_capita["capital"].apply(
    lambda x: "Maceió" if "Maceió" in x else "Outras capitais"
)

df_per_capita.sort_values(["funcao", "per_capita"], ascending=[True, False])

,capital,uf,funcao,populacao,valor_pago,per_capita,destaque
26,Prefeitura Municipal de Belo Horizonte - MG,MG,10 - Saúde,2392678,5.391130e+09,2253.178065,Outras capitais
35,Prefeitura Municipal de Teresina - PI,PI,10 - Saúde,868523,1.740078e+09,2003.491285,Outras capitais
48,Prefeitura Municipal de Campo Grande - MS,MS,10 - Saúde,942140,1.750536e+09,1858.042309,Outras capitais
39,Prefeitura Municipal de São Paulo - SP,SP,10 - Saúde,12200180,2.185446e+10,1791.323132,Outras capitais
18,Prefeitura Municipal de Porto Alegre - RS,RS,10 - Saúde,1404269,2.443203e+09,1739.839635,Outras capitais
21,Prefeitura Municipal de Natal - RN,RN,10 - Saúde,751932,1.276230e+09,1697.267821,Outras capitais
10,Prefeitura Municipal de Curitiba - PR,PR,10 - Saúde,1871789,3.029796e+09,1618.663342,Outras capitais
45,Prefeitura Municipal de Goiânia - GO,GO,10 - Saúde,1414483,2.068846e+09,1462.616167,Outras capitais
40,Prefeitura Municipal de Cuiabá - MT,MT,10 - Saúde,694244,1.007145e+09,1450.708173,Outras capitais
41,Prefeitura Municipal de Fortaleza - CE,CE,10 - Saúde,2596157,3.508508e+09,1351.423765,Outras capitais


In [8]:
def grafico_per_capita(funcao_nome, titulo):
    dados = df_per_capita[df_per_capita["funcao"] == funcao_nome].sort_values("per_capita")
    ordem_capitais = dados["capital"].tolist()  # ascendente: menor per capita embaixo, maior em cima

    fig = px.bar(
        dados,
        x="per_capita",
        y="capital",
        color="destaque",
        orientation="h",
        color_discrete_map={"Maceió": "crimson", "Outras capitais": "lightslategray"},
        category_orders={"capital": ordem_capitais},  # <- fixa a ordem certa
        title=titulo,
        labels={"per_capita": "R$ por habitante", "capital": ""},
    )
    fig.update_layout(height=750, showlegend=False)
    return fig

grafico_per_capita("10 - Saúde", "Gasto per capita em Saúde (Despesas Pagas), 2024").show()

In [9]:
grafico_per_capita("12 - Educação", "Gasto per capita em Educação (Despesas Pagas), 2024").show()

## Conclusões — Execução Financeira e Gasto per Capita (2024)

Maceió apresenta padrões distintos dependendo da área analisada:

- **Habitação**: menor taxa de execução entre todas as combinações capital-função 
  do estudo (30%, contra 85% de média nas demais capitais). O gasto está 
  concentrado numa única subfunção (16.482 - Habitação Urbana), sugerindo que o 
  problema está ligado a um projeto/contrato específico, não a uma falha 
  generalizada de planejamento orçamentário.

- **Educação**: taxa de execução dentro do esperado, mas o valor per capita pago 
  (R$ 715,71/habitante) é o segundo mais baixo entre as 26 capitais — indicando 
  que o problema aqui não é executar o que foi orçado, e sim o tamanho do 
  orçamento per capita desde a origem.

- **Saúde**: posição mediana (13ª de 26) tanto em execução quanto em per capita, 
  sem padrão de destaque.

**Ressalva:** estes indicadores apontam padrões que merecem investigação mais 
aprofundada, não conclusões definitivas sobre eficiência ou desperdício — fatores 
como qualidade do serviço prestado e necessidades específicas de cada capital não 
estão presentes neste dataset.

In [10]:
query_execucao_geral = """
    WITH base AS (
        SELECT
            "Instituição" AS capital,
            funcao,
            "Coluna" AS estagio,
            SUM("Valor") AS valor
        FROM despesas
        WHERE tipo_conta = 'funcao'
          AND ano = 2024
          AND "Coluna" IN ('Despesas Empenhadas', 'Despesas Pagas')
        GROUP BY 1, 2, 3
    ),
    execucao_por_capital AS (
        SELECT
            capital,
            funcao,
            MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END) AS empenhado,
            MAX(CASE WHEN estagio = 'Despesas Pagas' THEN valor END) AS pago,
            100.0 * MAX(CASE WHEN estagio = 'Despesas Pagas' THEN valor END)
              / NULLIF(MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END), 0) AS taxa_execucao_pct
        FROM base
        GROUP BY capital, funcao
    )
    SELECT
        funcao,
        COUNT(*) AS n_capitais,
        ROUND(AVG(taxa_execucao_pct), 1) AS taxa_media,
        ROUND(MIN(taxa_execucao_pct), 1) AS taxa_minima,
        ROUND(MAX(taxa_execucao_pct), 1) AS taxa_maxima,
        ROUND(STDDEV(taxa_execucao_pct), 1) AS desvio_padrao
    FROM execucao_por_capital
    WHERE taxa_execucao_pct IS NOT NULL
    GROUP BY funcao
    ORDER BY taxa_media ASC
"""

df_execucao_geral = con.execute(query_execucao_geral).df()
df_execucao_geral

,funcao,n_capitais,taxa_media,taxa_minima,taxa_maxima,desvio_padrao
0,20 - Agricultura,15,83.8,50.3,100.0,17.3
1,16 - Habitação,24,85.2,30.0,100.0,17.4
2,23 - Comércio e Serviços,25,86.4,29.9,100.0,16.3
3,17 - Saneamento,24,87.2,32.9,100.0,15.9
4,24 - Comunicações,8,87.7,51.2,100.0,17.4
5,13 - Cultura,26,88.5,69.6,100.0,8.6
6,15 - Urbanismo,26,88.6,54.4,99.9,10.0
7,14 - Direitos da Cidadania,24,88.8,59.7,100.0,11.5
8,27 - Desporto e Lazer,26,89.6,72.4,98.6,8.2
9,18 - Gestão Ambiental,26,89.7,40.3,100.0,13.9


In [11]:
df_filtrado = df_execucao_geral[df_execucao_geral["n_capitais"] >= 15].copy()

fig = px.scatter(
    df_filtrado,
    x="taxa_media",
    y="desvio_padrao",
    text="funcao",
    size="n_capitais",
    color="taxa_media",
    color_continuous_scale="RdYlGn",
    title="Taxa média de execução vs. variação entre capitais (2024, funções com ≥15 capitais)",
    labels={"taxa_media": "Taxa de execução média (%)", "desvio_padrao": "Desvio-padrão entre capitais (p.p.)"},
)
fig.update_traces(textposition="top center")
fig.update_layout(height=650)
fig.show()

df_filtrado.sort_values("taxa_media")

,funcao,n_capitais,taxa_media,taxa_minima,taxa_maxima,desvio_padrao
0,20 - Agricultura,15,83.8,50.3,100.0,17.3
1,16 - Habitação,24,85.2,30.0,100.0,17.4
2,23 - Comércio e Serviços,25,86.4,29.9,100.0,16.3
3,17 - Saneamento,24,87.2,32.9,100.0,15.9
5,13 - Cultura,26,88.5,69.6,100.0,8.6
6,15 - Urbanismo,26,88.6,54.4,99.9,10.0
7,14 - Direitos da Cidadania,24,88.8,59.7,100.0,11.5
8,27 - Desporto e Lazer,26,89.6,72.4,98.6,8.2
9,18 - Gestão Ambiental,26,89.7,40.3,100.0,13.9
10,06 - Segurança Pública,25,92.1,65.9,100.0,8.3


## Conclusões — Padrão de Execução por Tipo de Despesa (2024, todas as capitais)

Existe uma relação clara entre o tipo de despesa e seu padrão de execução:

- **Despesas de investimento/infraestrutura** (Habitação, Saneamento, Agricultura, 
  Comércio e Serviços) têm a menor taxa média de execução (84-87%) e a maior 
  variação entre capitais (desvio-padrão de 15-17 p.p.). Isso indica que a 
  execução nessas áreas depende fortemente de fatores específicos de cada 
  gestão — como andamento de obras, licitações e contratos — e não é uma 
  limitação estrutural do setor público municipal como um todo.

- **Despesas continuadas e obrigatórias** (Previdência Social, Encargos 
  Especiais, Legislativa, Saúde) têm taxa média alta (96-99%) e baixíssima 
  variação (desvio-padrão de 2-5 p.p.), por dependerem de compromissos fixos 
  recorrentes, com pouca margem de planejamento discricionário.

- O caso de **Maceió em Habitação** (visto no eixo anterior) se encaixa 
  exatamente nesse padrão: é justamente na categoria de maior variação entre 
  capitais que aparece a maior distância entre uma capital específica e a média.

**Ressalva:** funções com poucas capitais na amostra (n < 15, ex.: Relações 
Exteriores, Defesa Nacional, Energia) foram excluídas deste gráfico por baixa 
representatividade estatística.

In [12]:
query_evolucao = """
    SELECT
        ano,
        "Instituição" AS capital,
        funcao,
        MAX("População") AS populacao,
        SUM("Valor") AS valor_pago
    FROM despesas
    WHERE tipo_conta = 'funcao'
      AND ano BETWEEN 2020 AND 2024
      AND "Coluna" = 'Despesas Pagas'
      AND funcao IN ('10 - Saúde', '12 - Educação')
    GROUP BY ano, capital, funcao
"""

df_evolucao = con.execute(query_evolucao).df()
df_evolucao["per_capita"] = df_evolucao["valor_pago"] / df_evolucao["populacao"]

# Média das capitais por ano/função
media_anual = (
    df_evolucao.groupby(["ano", "funcao"])["per_capita"]
    .mean()
    .reset_index()
    .assign(serie="Média das capitais")
)

# Recorte de Maceió
maceio_anual = (
    df_evolucao[df_evolucao["capital"].str.contains("Maceió")]
    [["ano", "funcao", "per_capita"]]
    .assign(serie="Maceió")
)

df_comparativo = pd.concat([media_anual, maceio_anual], ignore_index=True)
df_comparativo

,ano,funcao,per_capita,serie
0,2020,10 - Saúde,881.199696,Média das capitais
1,2020,12 - Educação,617.600685,Média das capitais
2,2021,10 - Saúde,939.081746,Média das capitais
3,2021,12 - Educação,659.025495,Média das capitais
4,2022,10 - Saúde,997.614655,Média das capitais
5,2022,12 - Educação,835.777466,Média das capitais
6,2023,10 - Saúde,1158.779082,Média das capitais
7,2023,12 - Educação,957.602247,Média das capitais
8,2024,10 - Saúde,1348.421756,Média das capitais
9,2024,12 - Educação,1158.053072,Média das capitais


In [13]:
df_comparativo_sorted = df_comparativo.sort_values(["funcao", "serie", "ano"])

fig = px.line(
    df_comparativo_sorted,
    x="ano",
    y="per_capita",
    color="serie",
    facet_col="funcao",
    markers=True,
    color_discrete_map={"Maceió": "crimson", "Média das capitais": "lightslategray"},
    title="Evolução do gasto per capita (Despesas Pagas) — Maceió vs. média das capitais, 2020-2024",
    labels={"per_capita": "R$ por habitante", "ano": "Ano", "serie": ""},
)
fig.update_layout(height=500)
fig.update_yaxes(matches=None)
fig.show()

In [14]:
pivot = df_comparativo.pivot_table(index=["ano", "funcao"], columns="serie", values="per_capita").reset_index()
pivot["gap_pct"] = 100 * (pivot["Maceió"] - pivot["Média das capitais"]) / pivot["Média das capitais"]
pivot.sort_values(["funcao", "ano"])

serie,ano,funcao,Maceió,Média das capitais,gap_pct
0,2020,10 - Saúde,766.940710,881.199696,-12.966299
2,2021,10 - Saúde,737.287469,939.081746,-21.488468
4,2022,10 - Saúde,832.413497,997.614655,-16.559616
6,2023,10 - Saúde,1235.461297,1158.779082,6.617501
8,2024,10 - Saúde,1314.665278,1348.421756,-2.503407
1,2020,12 - Educação,313.160082,617.600685,-49.294084
3,2021,12 - Educação,375.401949,659.025495,-43.036809
5,2022,12 - Educação,702.251440,835.777466,-15.976265
7,2023,12 - Educação,593.797342,957.602247,-37.991234
9,2024,12 - Educação,715.711584,1158.053072,-38.196996


## Conclusões — Evolução do Gasto per Capita (2020-2024)

- **Saúde**: o gap entre Maceió e a média das capitais oscilou nos últimos 5 anos, 
  passando de -21% (2021) para +6,6% acima da média em 2023, e voltando a -2,5% 
  em 2024. Nos últimos dois anos, Maceió está próxima da média das capitais nesta 
  função — diferente do padrão mais estável observado em Educação.

- **Educação**: o gap permanece consistentemente negativo e substancial ao longo 
  de todo o período (entre -38% e -49%), com uma melhora moderada (de -49% em 
  2020 para -38% em 2024). Apesar do valor per capita ter crescido em termos 
  absolutos, a diferença relativa em relação à média das capitais segue expressiva, 
  reforçando o achado do eixo anterior: Educação é uma área onde Maceió consistentemente 
  investe menos por habitante que a maioria das demais capitais.

**Ressalva:** os anos de 2020-2021 coincidem com o período agudo da pandemia de 
COVID-19, que pode ter distorcido tanto os valores de Saúde (aumento emergencial 
de gastos) quanto a execução geral do orçamento em todas as capitais — a leitura 
da série deve considerar esse contexto.

In [15]:
query_peso_administracao = """
    WITH gasto_por_capital AS (
        SELECT
            "Instituição" AS capital,
            funcao,
            SUM("Valor") AS total_pago
        FROM despesas
        WHERE tipo_conta = 'funcao'
          AND ano = 2024
          AND "Coluna" = 'Despesas Pagas'
        GROUP BY capital, funcao
    ),
    total_por_capital AS (
        SELECT capital, SUM(total_pago) AS total_geral
        FROM gasto_por_capital
        GROUP BY capital
    )
    SELECT
        g.capital,
        t.total_geral,
        g.total_pago AS gasto_administracao,
        ROUND(100.0 * g.total_pago / t.total_geral, 2) AS pct_administracao
    FROM gasto_por_capital g
    JOIN total_por_capital t ON g.capital = t.capital
    WHERE g.funcao = '04 - Administração'
    ORDER BY pct_administracao DESC
"""

df_administracao = con.execute(query_peso_administracao).df()
df_administracao

,capital,total_geral,gasto_administracao,pct_administracao
0,Prefeitura Municipal de Porto Velho - RO,2.476436e+09,5.568045e+08,22.48
1,Prefeitura Municipal de Goiânia - GO,8.858801e+09,1.719294e+09,19.41
2,Prefeitura Municipal de Macapá - AP,2.061415e+09,3.830764e+08,18.58
3,Prefeitura Municipal de Florianópolis - SC,3.373053e+09,5.394289e+08,15.99
4,Prefeitura Municipal de Maceió - AL,4.699869e+09,7.139572e+08,15.19
5,Prefeitura Municipal de Cuiabá - MT,3.493769e+09,5.287822e+08,15.14
6,Prefeitura Municipal de João Pessoa - PB,4.146367e+09,5.680839e+08,13.70
7,Prefeitura Municipal de Aracaju - SE,3.221706e+09,3.912291e+08,12.14
8,Prefeitura Municipal de Teresina - PI,4.870576e+09,5.154341e+08,10.58
9,Prefeitura Municipal de Boa Vista - RR,2.246982e+09,2.215443e+08,9.86


## Conclusões — Peso da Administração no Orçamento (2024)

Maceió destina 15,19% do seu gasto total (Despesas Pagas) à função 
04 - Administração, a 5ª maior proporção entre as 26 capitais e bem acima da 
média do grupo (~9,9%). O intervalo entre capitais é amplo (2,51% a 22,48%), 
o que, combinado com o achado do gasto per capita relativamente baixo em 
Educação, sugere uma proporção do orçamento comparativamente maior alocada 
à estrutura administrativa em relação a algumas áreas finalísticas.

**Ressalva:** a grande variação entre capitais nesse indicador provavelmente 
reflete, em parte, diferenças na forma como cada prefeitura classifica 
contabilmente seus gastos (ex.: alocar TI/RH dentro de Administração vs. 
distribuir esses custos nas funções finalísticas), e não apenas diferenças 
reais de eficiência de gestão. Este indicador deve ser lido como um sinal 
para investigação mais aprofundada, não como conclusão definitiva.

In [16]:
con.execute("SELECT DISTINCT \"Coluna\" FROM despesas ORDER BY 1").df()

,Coluna
0,Despesas Empenhadas
1,Despesas Liquidadas
2,Despesas Pagas
3,Inscrição de Restos a Pagar Não Processados
4,Inscrição de Restos a Pagar Processados


In [17]:
query_restos_a_pagar = """
    WITH base AS (
        SELECT
            "Instituição" AS capital,
            "UF" AS uf,
            funcao,
            "Coluna" AS estagio,
            SUM("Valor") AS valor
        FROM despesas
        WHERE tipo_conta = 'funcao'
          AND ano = 2024
          AND "Coluna" IN ('Despesas Empenhadas', 'Inscrição de Restos a Pagar Não Processados')
        GROUP BY 1, 2, 3, 4
    )
    SELECT
        capital,
        uf,
        funcao,
        MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END) AS empenhado,
        MAX(CASE WHEN estagio = 'Inscrição de Restos a Pagar Não Processados' THEN valor END) AS restos_nao_processados,
        ROUND(
            100.0 * MAX(CASE WHEN estagio = 'Inscrição de Restos a Pagar Não Processados' THEN valor END)
            / NULLIF(MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END), 0),
            1
        ) AS pct_travado_sem_liquidar
    FROM base
    GROUP BY capital, uf, funcao
    HAVING MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END) > 0
    ORDER BY pct_travado_sem_liquidar DESC
"""

df_restos = con.execute(query_restos_a_pagar).df()
df_restos.head(20)

,capital,uf,funcao,empenhado,restos_nao_processados,pct_travado_sem_liquidar
0,Prefeitura Municipal de Florianópolis - SC,SC,23 - Comércio e Serviços,2.257400e+05,2.257400e+05,100.0
1,Prefeitura Municipal de São Luís - MA,MA,24 - Comunicações,3.600000e+05,3.600000e+05,100.0
2,Prefeitura Municipal de Maceió - AL,AL,16 - Habitação,2.168153e+06,1.517387e+06,70.0
3,Prefeitura Municipal de Teresina - PI,PI,17 - Saneamento,4.825483e+06,2.986596e+06,61.9
4,Prefeitura Municipal de Teresina - PI,PI,18 - Gestão Ambiental,3.913655e+06,2.277598e+06,58.2
5,Prefeitura Municipal de Curitiba - PR,PR,16 - Habitação,4.745052e+07,2.564289e+07,54.0
6,Prefeitura Municipal de São Luís - MA,MA,23 - Comércio e Serviços,1.867306e+06,1.003449e+06,53.7
7,Prefeitura Municipal de Vitória - ES,ES,23 - Comércio e Serviços,1.573912e+07,7.934637e+06,50.4
8,Prefeitura Municipal de Vitória - ES,ES,24 - Comunicações,9.937871e+06,4.693838e+06,47.2
9,Prefeitura Municipal de Curitiba - PR,PR,15 - Urbanismo,1.805831e+09,8.059139e+08,44.6


In [18]:
LIMITE_MINIMO = 1_000_000  # R$ 1 milhão — abaixo disso, % fica instável e pouco representativo

df_restos_filtrado = df_restos[df_restos["empenhado"] >= LIMITE_MINIMO].copy()
df_restos_filtrado.head(20)

,capital,uf,funcao,empenhado,restos_nao_processados,pct_travado_sem_liquidar
2,Prefeitura Municipal de Maceió - AL,AL,16 - Habitação,2.168153e+06,1.517387e+06,70.0
3,Prefeitura Municipal de Teresina - PI,PI,17 - Saneamento,4.825483e+06,2.986596e+06,61.9
4,Prefeitura Municipal de Teresina - PI,PI,18 - Gestão Ambiental,3.913655e+06,2.277598e+06,58.2
5,Prefeitura Municipal de Curitiba - PR,PR,16 - Habitação,4.745052e+07,2.564289e+07,54.0
6,Prefeitura Municipal de São Luís - MA,MA,23 - Comércio e Serviços,1.867306e+06,1.003449e+06,53.7
7,Prefeitura Municipal de Vitória - ES,ES,23 - Comércio e Serviços,1.573912e+07,7.934637e+06,50.4
8,Prefeitura Municipal de Vitória - ES,ES,24 - Comunicações,9.937871e+06,4.693838e+06,47.2
9,Prefeitura Municipal de Curitiba - PR,PR,15 - Urbanismo,1.805831e+09,8.059139e+08,44.6
10,Prefeitura Municipal de Porto Velho - RO,RO,26 - Transporte,2.382498e+06,1.004621e+06,42.2
13,Prefeitura Municipal de Vitória - ES,ES,14 - Direitos da Cidadania,1.318704e+07,5.176720e+06,39.3


## Conclusões — Restos a Pagar Não Processados (2024)

Além da taxa de execução (Pago/Empenhado), analisar o quanto do valor empenhado 
sequer chegou a ser liquidado (Restos a Pagar Não Processados) revela um nível 
mais granular do mesmo problema. Para evitar distorções, o ranking abaixo 
considera apenas combinações capital-função com valor empenhado igual ou 
superior a R$ 1 milhão — abaixo disso, pequenas variações em valores absolutos 
baixos geram percentuais instáveis e pouco representativos.

- **Habitação (Maceió)**: 70% do valor empenhado em 2024 ficou como restos a 
  pagar não processados — o dinheiro não apenas não foi pago, como nem chegou 
  à fase de conferência/liquidação. A análise temporal (2020-2024) mostra que 
  esse padrão não é constante: a execução em Habitação varia de forma extrema 
  ano a ano (ver seção "Habitação e Administração ao Longo do Tempo"), sugerindo 
  que cada exercício está sujeito a um projeto/contrato pontual e independente, 
  com resultado imprevisível, em vez de um problema crônico e único paralisado.

- **Segurança Pública (33,6%) e Direitos da Cidadania (30,8%)**: essas mesmas 
  funções já haviam aparecido entre as piores no Eixo 1 (taxa de execução abaixo 
  da média das capitais). A convergência entre as duas métricas reforça que o 
  problema nessas áreas é consistente, não um artefato de uma métrica isolada.

- Maceió aparece 3 vezes neste ranking filtrado (empatada com Curitiba, Vitória 
  e Porto Velho como as capitais com mais ocorrências) — sugerindo que a 
  dificuldade de fazer o orçamento avançar até a liquidação não está restrita a 
  uma única área, mas se repete em funções distintas.

In [19]:
query_habitacao_temporal = """
    WITH base AS (
        SELECT
            "Instituição" AS capital,
            ano,
            "Coluna" AS estagio,
            SUM("Valor") AS valor
        FROM despesas
        WHERE tipo_conta = 'funcao'
          AND funcao = '16 - Habitação'
          AND ano BETWEEN 2020 AND 2024
          AND "Coluna" IN ('Despesas Empenhadas', 'Despesas Pagas')
          AND "Instituição" LIKE '%Maceió%'
        GROUP BY 1, 2, 3
    )
    SELECT
        ano,
        MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END) AS empenhado,
        MAX(CASE WHEN estagio = 'Despesas Pagas' THEN valor END) AS pago,
        ROUND(
            100.0 * MAX(CASE WHEN estagio = 'Despesas Pagas' THEN valor END)
            / NULLIF(MAX(CASE WHEN estagio = 'Despesas Empenhadas' THEN valor END), 0),
            1
        ) AS taxa_execucao_pct
    FROM base
    GROUP BY ano
    ORDER BY ano
"""

df_habitacao_temporal = con.execute(query_habitacao_temporal).df()
df_habitacao_temporal

,ano,empenhado,pago,taxa_execucao_pct
0,2020,3413025.41,1023786.88,30.0
1,2021,929063.67,305.50,0.0
2,2022,8705136.85,8652072.30,99.4
3,2023,508619.44,438254.98,86.2
4,2024,2168153.21,649674.84,30.0


In [20]:
query_administracao_temporal = """
    WITH gasto_por_capital_ano AS (
        SELECT
            "Instituição" AS capital,
            ano,
            funcao,
            SUM("Valor") AS total_pago
        FROM despesas
        WHERE tipo_conta = 'funcao'
          AND ano BETWEEN 2020 AND 2024
          AND "Coluna" = 'Despesas Pagas'
        GROUP BY capital, ano, funcao
    ),
    total_por_capital_ano AS (
        SELECT capital, ano, SUM(total_pago) AS total_geral
        FROM gasto_por_capital_ano
        GROUP BY capital, ano
    ),
    pct_administracao AS (
        SELECT
            g.capital,
            g.ano,
            ROUND(100.0 * g.total_pago / t.total_geral, 2) AS pct_administracao
        FROM gasto_por_capital_ano g
        JOIN total_por_capital_ano t ON g.capital = t.capital AND g.ano = t.ano
        WHERE g.funcao = '04 - Administração'
    )
    SELECT
        ano,
        AVG(CASE WHEN capital LIKE '%Maceió%' THEN pct_administracao END) AS maceio,
        AVG(pct_administracao) AS media_capitais
    FROM pct_administracao
    GROUP BY ano
    ORDER BY ano
"""

df_administracao_temporal = con.execute(query_administracao_temporal).df()
df_administracao_temporal

,ano,maceio,media_capitais
0,2020,16.00,10.621154
1,2021,15.37,10.252692
2,2022,13.70,10.318077
3,2023,14.42,9.923846
4,2024,15.19,9.525000


In [21]:
fig1 = px.line(
    df_habitacao_temporal,
    x="ano",
    y="taxa_execucao_pct",
    markers=True,
    title="Taxa de execução em Habitação — Maceió, 2020-2024 (volatilidade ano a ano)",
    labels={"taxa_execucao_pct": "Taxa de execução (%)", "ano": "Ano"},
)
fig1.add_hline(y=85.2, line_dash="dash", line_color="gray",
                annotation_text="Média das capitais em 2024 (85,2%)")
fig1.update_traces(line_color="crimson", line_width=3, marker_size=10)
fig1.update_layout(height=450, yaxis_range=[0, 105])
fig1.show()

In [22]:
df_admin_melt = df_administracao_temporal.melt(
    id_vars="ano", value_vars=["maceio", "media_capitais"],
    var_name="serie", value_name="pct_administracao"
)
df_admin_melt["serie"] = df_admin_melt["serie"].map({"maceio": "Maceió", "media_capitais": "Média das capitais"})

fig2 = px.line(
    df_admin_melt,
    x="ano",
    y="pct_administracao",
    color="serie",
    markers=True,
    color_discrete_map={"Maceió": "crimson", "Média das capitais": "lightslategray"},
    title="Peso da Administração no orçamento — Maceió vs. média das capitais, 2020-2024",
    labels={"pct_administracao": "% do gasto total", "ano": "Ano", "serie": ""},
)
fig2.update_layout(height=450)
fig2.show()

## Conclusões — Habitação e Administração ao Longo do Tempo (2020-2024)

**Habitação: volatilidade, não recorrência.** A taxa de execução em Maceió 
oscilou de forma extrema entre 2020 e 2024: 30% → 0% → 99,4% → 86,2% → 30%. 
Diferente da hipótese inicial de um problema estrutural contínuo, o padrão 
real é de forte instabilidade ano a ano, sugerindo que o gasto em Habitação 
depende de projetos/contratos pontuais e não vinculados a um programa contínuo 
e previsível — em alguns anos o projeto avança quase integralmente (2022, 2023), 
em outros praticamente não sai do papel (2021, 2024).

**Administração: um padrão estrutural persistente.** Ao contrário de Habitação, 
o peso da Administração no orçamento de Maceió é consistentemente mais alto que 
a média das capitais em todos os 5 anos analisados (diferença entre +3,4 e +5,7 
pontos percentuais, sem exceção). Essa persistência ao longo do tempo torna esse 
achado mais robusto que uma comparação de ano único — não é uma anomalia de 
2024, é um traço estrutural da alocação orçamentária de Maceió.

**Leitura conjunta:** os dois indicadores revelam problemas de natureza 
diferente. Habitação sofre de **imprevisibilidade de execução** (o problema 
é *quando* o dinheiro sai, não *quanto* é destinado). Administração sofre de 
**alocação estruturalmente elevada** (o problema é *quanto* é destinado, de 
forma consistente, a um item que não é finalístico).

In [23]:
print(f"Categorias tratadas como 'agregado' (excluídas das somas): {df[df['tipo_conta'] == 'agregado']['Conta'].nunique()}")
df[df["tipo_conta"] == "agregado"]["Conta"].unique()

Categorias tratadas como 'agregado' (excluídas das somas): 28


<ArrowStringArray>
['Despesas Exceto Intraorçamentárias',           'FU08 - Demais Subfunções',
           'FU11 - Demais Subfunções',           'FU20 - Demais Subfunções',
           'FU26 - Demais Subfunções',        'Despesas Intraorçamentárias',
           'FU01 - Demais Subfunções',           'FU04 - Demais Subfunções',
           'FU09 - Demais Subfunções',           'FU10 - Demais Subfunções',
           'FU12 - Demais Subfunções',           'FU27 - Demais Subfunções',
           'FU02 - Demais Subfunções',           'FU06 - Demais Subfunções',
           'FU13 - Demais Subfunções',           'FU14 - Demais Subfunções',
           'FU16 - Demais Subfunções',           'FU17 - Demais Subfunções',
           'FU18 - Demais Subfunções',           'FU03 - Demais Subfunções',
           'FU15 - Demais Subfunções',           'FU19 - Demais Subfunções',
           'FU23 - Demais Subfunções',           'FU24 - Demais Subfunções',
           'FU28 - Demais Subfunções',           'FU22 - 

In [24]:
query_intraorcamentaria = """
    SELECT
        "Instituição" AS capital,
        ano,
        "Coluna" AS estagio,
        SUM("Valor") AS valor
    FROM despesas
    WHERE "Conta" = 'Despesas Intraorçamentárias'
      AND ano = 2024
      AND "Coluna" = 'Despesas Pagas'
    GROUP BY "Instituição", ano, "Coluna"
    ORDER BY valor DESC
    LIMIT 10
"""

df_intra = con.execute(query_intraorcamentaria).df()
df_intra

,capital,ano,estagio,valor
0,Prefeitura Municipal de São Paulo - SP,2024,Despesas Pagas,1.133185e+10
1,Prefeitura Municipal do Rio de Janeiro - RJ,2024,Despesas Pagas,6.646994e+09
2,Prefeitura Municipal de Curitiba - PR,2024,Despesas Pagas,1.697930e+09
3,Prefeitura Municipal de Porto Alegre - RS,2024,Despesas Pagas,1.619899e+09
4,Prefeitura Municipal de Belo Horizonte - MG,2024,Despesas Pagas,1.159363e+09
5,Prefeitura Municipal de Fortaleza - CE,2024,Despesas Pagas,8.755407e+08
6,Prefeitura Municipal de Salvador - BA,2024,Despesas Pagas,4.461359e+08
7,Prefeitura Municipal de Goiânia - GO,2024,Despesas Pagas,4.412371e+08
8,Prefeitura Municipal de Recife - PE,2024,Despesas Pagas,4.348829e+08
9,Prefeitura Municipal de Manaus - AM,2024,Despesas Pagas,4.245233e+08


In [25]:
query_comparativo_intra = """
    SELECT
        "Instituição" AS capital,
        "Conta",
        SUM("Valor") AS valor
    FROM despesas
    WHERE "Conta" IN ('Despesas Exceto Intraorçamentárias', 'Despesas Intraorçamentárias')
      AND ano = 2024
      AND "Coluna" = 'Despesas Pagas'
      AND "Instituição" LIKE '%Maceió%'
    GROUP BY "Instituição", "Conta"
"""

con.execute(query_comparativo_intra).df()

,capital,Conta,valor
0,Prefeitura Municipal de Maceió - AL,Despesas Exceto Intraorçamentárias,4.699869e+09
1,Prefeitura Municipal de Maceió - AL,Despesas Intraorçamentárias,2.160709e+08


**Verificação — Despesas Intraorçamentárias:** confirmado que representam uma 
fração pequena do orçamento (ex.: Maceió, 2024: R$ 216,1 milhões de 
Intraorçamentárias vs. R$ 4,70 bilhões de Exceto Intraorçamentárias, ~4,6%), 
consistente com o conceito de "um órgão público pagando outro do mesmo 
município". Essas linhas foram corretamente excluídas das somas por 
função/subfunção (classificadas como `tipo_conta = 'agregado'`), evitando 
dupla contagem.